# AI Tutor on Kaggle

1. Chọn **Copy & Edit**, bật **GPU accelerator** và **Internet**.
2. Attach Kaggle Dataset/Model chứa file GGUF; attach dataset bài giảng nếu cần.
3. Tạo Kaggle Secret tên `NGROK_AUTHTOKEN`.
4. (Tuỳ chọn nhưng khuyến nghị) Để giữ lại đoạn chat/quiz và chạy nhanh hơn ở các lần sau: tạo thêm 2 Kaggle Secret `KAGGLE_USERNAME` và `KAGGLE_KEY` (lấy từ kaggle.com/settings → **Create New Token**, mở file `kaggle.json` tải về để lấy 2 giá trị này). Sau đó điền `STATE_DATASET_SLUG` (bắt buộc để lưu chat/quiz) và `CACHE_DATASET_SLUG` (tuỳ chọn, để cache model/package cho khởi động nhanh) trong cell **User configuration**. Dataset không cần tạo trước, lần lưu đầu tiên sẽ tự tạo (ở chế độ riêng tư).
5. Sửa duy nhất cell **User configuration**, sau đó chọn **Run All**.
6. Mở `AI Tutor URL` được in ở cell tunnel gần cuối notebook.
7. Trước khi dừng session (hoặc bất cứ lúc nào muốn chốt tiến trình), xuống cuối notebook, đặt `SAVE_STATE_NOW = True` trong cell **Save progress** rồi chạy lại cell đó để lưu chat/quiz lên Kaggle Dataset. Lần host sau sẽ tự động khôi phục lại.</cell id="cell-0">

In [ ]:
import shutil, subprocess

for command in (["nvidia-smi"], ["python", "--version"], ["node", "--version"], ["npm", "--version"]):
    print("$", " ".join(command))
    if shutil.which(command[0]):
        subprocess.run(command, check=False)
    else:
        print(f"WARNING: {command[0]} is not installed")
if not shutil.which("nvidia-smi"):
    print("WARNING: GPU is not enabled. Select a GPU accelerator before running the model.")

In [ ]:
# User configuration: edit values only in this cell.
REPOSITORY_URL = "https://github.com/qtrung123/AI_Tutor2-Kaggle.git"
REPOSITORY_BRANCH = "main"
PROJECT_ROOT = "/kaggle/working/AI_Tutor2"
GGUF_MODEL_PATH = "/kaggle/input/datasets/trung121212/tutor-model/qwen2.5-7b-instruct-q4_k_m.gguf"
LECTURE_INPUT_DIR = ""  # Optional read-only directory under /kaggle/input
OLLAMA_CHAT_MODEL = "qwen-tutor-7b"
OLLAMA_EMBEDDING_MODEL = "bge-m3"
BACKEND_PORT = 8000
FRONTEND_PORT = 3000
PUBLIC_PORT = 7860
TUNNEL_PROVIDER = "ngrok"
RECREATE_OLLAMA_MODEL = False

# Persistence across Kaggle sessions (private Kaggle Datasets used as remote storage).
# Leave a slug empty to disable that part. Format: "kaggle-username/dataset-slug".
# The dataset does not need to exist beforehand; the first "Save" run creates it as private.
STATE_DATASET_SLUG = ""  # e.g. "trung121212/ai-tutor-state" -> keeps chat/quiz/documents
CACHE_DATASET_SLUG = ""  # e.g. "trung121212/ai-tutor-cache" -> keeps Ollama models + pip/npm cache

import os
for key, value in {
    "PROJECT_ROOT": PROJECT_ROOT, "GGUF_MODEL_PATH": GGUF_MODEL_PATH,
    "OLLAMA_CHAT_MODEL": OLLAMA_CHAT_MODEL,
    "OLLAMA_EMBEDDING_MODEL": OLLAMA_EMBEDDING_MODEL,
    "BACKEND_PORT": BACKEND_PORT, "FRONTEND_PORT": FRONTEND_PORT,
    "PUBLIC_PORT": PUBLIC_PORT,
    "RECREATE_OLLAMA_MODEL": int(RECREATE_OLLAMA_MODEL),
    "REBUILD_CHROMA_ON_EMBEDDING_CHANGE": 1,
}.items(): os.environ[key] = str(value)</cell id="cell-2">

In [ ]:
from pathlib import Path
import subprocess
root = Path(PROJECT_ROOT)
if not (root / ".git").exists():
    root.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "--branch", REPOSITORY_BRANCH, "--single-branch", REPOSITORY_URL, PROJECT_ROOT], check=True)
else:
    subprocess.run(["git", "-C", PROJECT_ROOT, "fetch", "origin", REPOSITORY_BRANCH], check=True)
    subprocess.run(["git", "-C", PROJECT_ROOT, "checkout", REPOSITORY_BRANCH], check=True)
    subprocess.run(["git", "-C", PROJECT_ROOT, "pull", "--ff-only", "origin", REPOSITORY_BRANCH], check=True)

## Restore previous session

Downloads chat/quiz/documents (`STATE_DATASET_SLUG`) and the Ollama model + package cache
(`CACHE_DATASET_SLUG`) from your private Kaggle Datasets, if configured and if a previous
save exists. Safe to run even on the very first session (nothing to restore yet).</cell id="restore-md">

In [ ]:
import os
import shutil
import sys
from pathlib import Path

sys.path.insert(0, f"{PROJECT_ROOT}/deployment")
import kaggle_persist

def _load_kaggle_credentials() -> bool:
    if os.getenv("KAGGLE_USERNAME") and os.getenv("KAGGLE_KEY"):
        return True
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        os.environ["KAGGLE_USERNAME"] = secrets.get_secret("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"] = secrets.get_secret("KAGGLE_KEY")
        return True
    except Exception as error:
        print("Add KAGGLE_USERNAME and KAGGLE_KEY as Kaggle Secrets to enable persistence.")
        print("Details:", error)
        return False

STAGING_DIR = Path("/kaggle/working/.persist")
CACHE_DIR = Path("/kaggle/working/.cache")
STAGING_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

PERSISTENCE_ENABLED = bool(STATE_DATASET_SLUG) and _load_kaggle_credentials()
CACHE_ENABLED = bool(CACHE_DATASET_SLUG) and _load_kaggle_credentials()

if PERSISTENCE_ENABLED:
    restore_dir = STAGING_DIR / "state_restore"
    if kaggle_persist.restore_dataset(STATE_DATASET_SLUG, restore_dir):
        for name in ("data", "vectorstore", "indexed_files.json"):
            source = restore_dir / name
            if not source.exists():
                continue
            target = Path(PROJECT_ROOT) / name
            if target.exists():
                shutil.rmtree(target) if target.is_dir() else target.unlink()
            shutil.move(str(source), str(target))
        print("Restored previous conversations, quizzes, and indexed documents.")
    else:
        print("Starting with empty data (no previous save found).")
else:
    print("State persistence disabled: set STATE_DATASET_SLUG and the KAGGLE_USERNAME/KAGGLE_KEY secrets to enable it.")

if CACHE_ENABLED:
    restore_dir = STAGING_DIR / "cache_restore"
    if kaggle_persist.restore_dataset(CACHE_DATASET_SLUG, restore_dir):
        for name in ("ollama-models", "pip", "npm"):
            source = restore_dir / name
            if not source.exists():
                continue
            target = CACHE_DIR / name
            if target.exists():
                shutil.rmtree(target)
            shutil.move(str(source), str(target))
        print("Restored the Ollama model + package cache.")
    else:
        print("No previous cache found; this run will populate one (save it with the cell at the end).")
else:
    print("Cache persistence disabled: set CACHE_DATASET_SLUG to skip re-downloading models/packages next time.")

(CACHE_DIR / "ollama-models").mkdir(parents=True, exist_ok=True)
(CACHE_DIR / "pip").mkdir(parents=True, exist_ok=True)
(CACHE_DIR / "npm").mkdir(parents=True, exist_ok=True)
os.environ["OLLAMA_MODELS"] = str(CACHE_DIR / "ollama-models")
os.environ["PIP_CACHE_DIR"] = str(CACHE_DIR / "pip")
os.environ["NPM_CONFIG_CACHE"] = str(CACHE_DIR / "npm")

In [ ]:
import shutil, subprocess
required_packages = [("nginx", "nginx"), ("curl", "curl"), ("zstd", "zstd")]
missing = [package for package, binary in required_packages if not shutil.which(binary)]
if missing:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", *missing], check=True)
if not shutil.which("ollama"):
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
subprocess.run(["ollama", "--version"], check=True)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-r", f"{PROJECT_ROOT}/requirements.txt"], check=True)

In [ ]:
import subprocess
frontend = f"{PROJECT_ROOT}/frontend"
subprocess.run(["npm", "ci", "--prefer-offline"], cwd=frontend, check=True)

In [ ]:
from pathlib import Path
import shutil
data_dir = Path(PROJECT_ROOT) / "data"
data_dir.mkdir(parents=True, exist_ok=True)
(Path(PROJECT_ROOT) / "vectorstore").mkdir(parents=True, exist_ok=True)
if LECTURE_INPUT_DIR:
    source_dir = Path(LECTURE_INPUT_DIR)
    if not source_dir.is_dir(): raise FileNotFoundError(f"Lecture input not found: {source_dir}")
    for source in source_dir.rglob("*"):
        if source.is_file() and source.suffix.lower() in {".pdf", ".txt"}:
            target = data_dir / source.name
            if not target.exists() or target.stat().st_size != source.stat().st_size:
                shutil.copy2(source, target)
print("Runtime data directory:", data_dir)

In [ ]:
import subprocess
subprocess.run(["bash", f"{PROJECT_ROOT}/deployment/start_kaggle.sh"], cwd=PROJECT_ROOT, check=True)
print("Health check:")
subprocess.run(["curl", "-fsS", f"http://127.0.0.1:{PUBLIC_PORT}/api/health"], check=True)
print("\nWarming up the chat model so GPU allocation can be verified...")
import httpx
warmup = httpx.post("http://127.0.0.1:11434/api/generate", json={"model": OLLAMA_CHAT_MODEL, "prompt": "Reply with OK.", "stream": False, "keep_alive": "10m"}, timeout=600)
warmup.raise_for_status()
print("Model response:", warmup.json().get("response", "").strip())
print("Checking the embedding model with non-private test text...")
embedding_warmup = httpx.post("http://127.0.0.1:11434/api/embed", json={"model": OLLAMA_EMBEDDING_MODEL, "input": "AI Tutor embedding health check.", "keep_alive": "10m"}, timeout=120)
embedding_warmup.raise_for_status()
if not embedding_warmup.json().get("embeddings"): raise RuntimeError("Embedding warm-up returned no vector.")
print("Embedding warm-up: OK")
print("ollama list / backend health / proxied health:")
subprocess.run(["ollama", "list"], check=True)
subprocess.run(["curl", "-fsS", f"http://127.0.0.1:{BACKEND_PORT}/api/health"], check=True)
print()
subprocess.run(["curl", "-fsS", f"http://127.0.0.1:{PUBLIC_PORT}/api/health"], check=True)
print("\nLoaded models / GPU check:")
subprocess.run(["ollama", "ps"], check=False)
subprocess.run(["nvidia-smi"], check=False)

In [ ]:
import subprocess, sys
if TUNNEL_PROVIDER != "ngrok":
    print(f"Unsupported tunnel provider: {TUNNEL_PROVIDER}")
else:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("NGROK_AUTHTOKEN")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyngrok"], check=True)
        from pyngrok import ngrok
        ngrok.kill()
        ngrok.set_auth_token(token)
        tunnel = ngrok.connect(addr=f"127.0.0.1:{PUBLIC_PORT}", proto="http")
        print(f"AI Tutor URL: {tunnel.public_url}")
    except Exception as error:
        print("Could not create the tunnel. Add a Kaggle Secret named NGROK_AUTHTOKEN, enable Internet, then rerun this cell.")
        print("Details:", error)

In [ ]:
import subprocess
for log_name in ["ollama.log", "backend.log", "frontend.log", "nginx.log"]:
    print(f"\n===== {log_name} =====")
    subprocess.run(["tail", "-n", "100", f"/kaggle/working/ai-tutor-logs/{log_name}"], check=False)
subprocess.run(["ollama", "list"], check=False)
subprocess.run(["ollama", "ps"], check=False)
subprocess.run(["nvidia-smi"], check=False)
ps = subprocess.run(["ollama", "ps"], capture_output=True, text=True, check=False).stdout
if ps.strip() and "GPU" not in ps.upper(): print("WARNING: Ollama does not report GPU usage; inspect PROCESSOR and nvidia-smi.")

## Save progress

Run this any time you want to snapshot chat/quiz/documents to `STATE_DATASET_SLUG` so the
next session can restore it. Set `SAVE_STATE_NOW = True` below and run this cell (do it again
whenever you want a newer snapshot, e.g. right before you stop the session). Leaving it `False`
lets `Run All` pass over this cell without uploading anything.

In [ ]:
SAVE_STATE_NOW = False  # Set True, then run this cell whenever you want to save progress.

if not SAVE_STATE_NOW:
    print("SAVE_STATE_NOW is False - nothing saved. Set it to True and rerun this cell to save.")
elif not PERSISTENCE_ENABLED:
    print("State persistence disabled: set STATE_DATASET_SLUG and Kaggle secrets, then rerun the restore cell.")
else:
    staging = STAGING_DIR / "state_save"
    if staging.exists():
        shutil.rmtree(staging)
    staging.mkdir(parents=True)
    for name in ("data", "vectorstore", "indexed_files.json"):
        source = Path(PROJECT_ROOT) / name
        if not source.exists():
            continue
        target = staging / name
        shutil.copytree(source, target) if source.is_dir() else shutil.copy2(source, target)
    kaggle_persist.save_dataset(
        STATE_DATASET_SLUG, staging,
        title="AI Tutor state",
        message="Snapshot from an active Kaggle session",
    )
    print("Progress saved. It will be restored automatically next time you run this notebook.")

## Save cache (optional, speeds up future sessions)

Run this once after a successful first setup (or after models/packages change) to snapshot the
Ollama models directory plus the pip/npm caches to `CACHE_DATASET_SLUG`. Future sessions restore
this instead of re-pulling the embedding model and reinstalling packages. This upload is a few
GB, so there's no need to run it every session - only when the cache actually changed.

In [ ]:
SAVE_CACHE_NOW = False  # Set True, then run this cell to snapshot the model/package cache.

if not SAVE_CACHE_NOW:
    print("SAVE_CACHE_NOW is False - nothing saved. Set it to True and rerun this cell to save.")
elif not CACHE_ENABLED:
    print("Cache persistence disabled: set CACHE_DATASET_SLUG and Kaggle secrets, then rerun the restore cell.")
else:
    staging = STAGING_DIR / "cache_save"
    if staging.exists():
        shutil.rmtree(staging)
    staging.mkdir(parents=True)
    for name in ("ollama-models", "pip", "npm"):
        source = CACHE_DIR / name
        if source.exists() and any(source.iterdir()):
            shutil.copytree(source, staging / name)
    kaggle_persist.save_dataset(
        CACHE_DATASET_SLUG, staging,
        title="AI Tutor build cache",
        message="Updated Ollama model + package cache",
    )
    print("Cache saved. Future sessions will skip re-downloading models and packages.")